# Study report: tick-mode

The study compares the default tickless kernel at 10000 ticks/s with
three variants: `ticks1000` (1000 ticks/s, still tickless),
`tickless-off` (a periodic tick at 10000 ticks/s) and
`tickless-off-100` (a periodic tick at 100 ticks/s). It asks if the
SysTick reprogram of the slice reset makes the two modes of the wake
and the clock drift (H1), and how the tick rate sets the IRQ tail (H2).
The study imports the three experiments of the parent study
kconfig-factors.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-kconfig-factors@nucleo_g474re/st-tick-mode"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "probatrix" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## Background

The study uses three mechanisms:

1. The SysTick reprogram. In the tickless kernel, the slice reset of a
   wake reprograms the SysTick. Without `TICKLESS_KERNEL`,
   `sys_clock_set_timeout()` returns at once (ws-up
   `drivers/timer/system_timer_generic.h:650`). Thus the slice reset
   never reprograms the SysTick.
2. The slice walk. The slice reset also walks the timeout list. This
   walk stays without the tickless kernel.
3. The tick ISR. Without the tickless kernel, a SysTick ISR comes at
   each tick: every 17000 cycles at 10000 ticks/s, every 1.7M cycles at
   100 ticks/s. At a lower tick rate, the kernel puts the expiries of
   the timers into fewer, longer tick ISRs.

The points `*-t64` have 64 load timers. Their reference is
`baseline-t64`.

In [ ]:
LOAD_NAMES = {"none": "no load", "busy16": "16 busy threads", "t64": "64 load timers"}


def diffs(exp):
    """Each point minus the point of the variant baseline with the same load, in the same run (nbfig.diffs)."""
    return nbfig.diffs(inst, exp, "variant", "baseline", match=("load",), names=LOAD_NAMES)

## How to read the difference figures

Each bar is the value of a point minus the value of the point with the
variant `baseline` and the same load, in the same run. A bar to the left
of 0 means that the variant is faster. The three panels give the p50,
the p99 and the max. Each panel has its own x scale. The colour is the
load of the point. The menu above the figure selects the metric.

A grey band is the range that a check of this study expects, in the
same unit. If a band has one open end, it comes from an `order` check.
The manual checks are in the table of the checks only. The table below
each figure gives the same differences in cycles.

## IRQ entry (irq-kconfig-1m)

H3 a expects no change of the `irq` p50 at 1000 ticks/s (+-2 cycles).
H2 b..e and h are `order` checks of the p99 and the max. H2 g expects
a max of 10000..20000 cycles for `tickless-off-100-t64`.

In [ ]:
VARIANTS = ['ticks1000', 'tickless-off', 'tickless-off-100']
if nbfig.built(inst, "irq-kconfig-1m"):
    d = diffs("irq-kconfig-1m")
    sel = d[d.variant.isin(VARIANTS)]
    nbfig.diff_fig(sel, "irq-kconfig-1m: each variant minus the baseline of the same load",
                   limits=nbfig.check_limits(inst, "irq-kconfig-1m", d)).show()

In [ ]:
nbfig.diff_table(sel) if nbfig.built(inst, "irq-kconfig-1m") else None

## ISR to thread (wake-kconfig-1m)

H1 a expects 250..300 cycles less for `tickless-off` (`isr2thr` p50).
H3 b expects less than 30 cycles of change at 1000 ticks/s.

In [ ]:
VARIANTS = ['ticks1000', 'tickless-off', 'tickless-off-100']
if nbfig.built(inst, "wake-kconfig-1m"):
    d = diffs("wake-kconfig-1m")
    sel = d[d.variant.isin(VARIANTS)]
    nbfig.diff_fig(sel, "wake-kconfig-1m: each variant minus the baseline of the same load",
                   limits=nbfig.check_limits(inst, "wake-kconfig-1m", d)).show()

In [ ]:
nbfig.diff_table(sel) if nbfig.built(inst, "wake-kconfig-1m") else None

## Thread to thread (thread-kconfig-1m)

H3 d expects less than 30 cycles of change of the `t2t` p50 at 1000
ticks/s.

In [ ]:
VARIANTS = ['ticks1000', 'tickless-off', 'tickless-off-100']
if nbfig.built(inst, "thread-kconfig-1m"):
    d = diffs("thread-kconfig-1m")
    sel = d[d.variant.isin(VARIANTS)]
    nbfig.diff_fig(sel, "thread-kconfig-1m: each variant minus the baseline of the same load",
                   limits=nbfig.check_limits(inst, "thread-kconfig-1m", d)).show()

In [ ]:
nbfig.diff_table(sel) if nbfig.built(inst, "thread-kconfig-1m") else None

## H1: the two modes of the wake and of the thread to thread latency

H1 says that the baseline has two modes: about 20 % fast and 80 % slow.
Without the tickless kernel, the latency has one mode, at the fast mode
of the baseline. H1 b and d..h are manual checks. The figures give the
fine bins of the histograms. The tables give the modes as
`check-forms.rst` defines them: a group of fine bins that each hold at
least 0.1 % of the samples.

In [ ]:
def xrange_of(exp, metric, points):
    df = inst.rows(exp)
    df = df[(df.metric == metric) & df.point.isin(points)]
    return (df["min"].min() * 0.97, df["p99.9"].max() * 1.03)


P0 = ["baseline", "ticks1000", "tickless-off", "tickless-off-100"]
P64 = ["baseline-t64", "ticks1000-t64", "tickless-off-t64", "tickless-off-100-t64"]
if nbfig.built(inst, "wake-kconfig-1m"):
    nbfig.hist(nbfig.hists_of(inst, "wake-kconfig-1m", points=P0), "wake-kconfig-1m: isr2thr, no load", ["isr2thr"],
               xrange=xrange_of("wake-kconfig-1m", "isr2thr", P0)).show()

In [ ]:
if nbfig.built(inst, "wake-kconfig-1m"):
    nbfig.hist(nbfig.hists_of(inst, "wake-kconfig-1m", points=P64), "wake-kconfig-1m: isr2thr, 64 load timers",
               ["isr2thr"],
               xrange=xrange_of("wake-kconfig-1m", "isr2thr", P64)).show()

In [ ]:
nbfig.mode_table(inst, ["wake-kconfig-1m"], "isr2thr", points=P0 + P64, max_modes=4) \
    if nbfig.built(inst, "wake-kconfig-1m") else None

In [ ]:
if nbfig.built(inst, "thread-kconfig-1m"):
    nbfig.hist(nbfig.hists_of(inst, "thread-kconfig-1m", points=P0), "thread-kconfig-1m: t2t", ["t2t"],
               xrange=xrange_of("thread-kconfig-1m", "t2t", P0)).show()

In [ ]:
nbfig.mode_table(inst, ["thread-kconfig-1m"], "t2t", points=P0, max_modes=4) \
    if nbfig.built(inst, "thread-kconfig-1m") else None

## H1 c, H3 c: the clock drift

The clock drift is LAT META `clock_drift_cyc`: the kernel cycle counter
against the stimulus timer. `drift_ppm` is `clock_drift_cyc` /
`clock_elapsed_cyc` x 1e6. Each panel is one experiment, with its own x
scale. H1 c expects -20..20 ppm for `tickless-off` (`wake-kconfig-1m`).
The grey band is this limit. H3 c (manual) expects
the same drift for each reprogram at 1000 ticks/s as in the baseline.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

exps = [e for e in ("irq-kconfig-1m", "wake-kconfig-1m", "thread-kconfig-1m") if nbfig.built(inst, e)]
dr = inst.drift(exps)
cats = list(dict.fromkeys(dr.point))
fig = make_subplots(rows=1, cols=len(exps), shared_yaxes=True, horizontal_spacing=0.03, subplot_titles=exps)
for k, e in enumerate(exps):
    sub = dr[dr.experiment == e]
    fig.add_trace(go.Bar(y=sub.point, x=sub.drift_ppm, orientation="h", name=e, marker_color=nbfig.color(k),
                         customdata=sub.drift_per_sample,
                         hovertemplate="%{y}: %{x:.1f} ppm (%{customdata:.3f} cycles per sample)<extra>" + e
                                       + "</extra>"), row=1, col=k + 1)
    fig.add_trace(go.Scatter(y=sub.point, x=sub.drift_ppm, mode="markers", showlegend=False, hoverinfo="skip",
                             marker={"color": nbfig.color(k), "size": 7, "line": {"color": "#fcfcfb", "width": 1}}),
                  row=1, col=k + 1)
    for h in inst.design_hypotheses.values():
        for c in h.get("checks", []):
            if c.get("stat") == "drift_ppm" and "range" in c.get("expect", {}) and c.get("experiment") == e:
                lo, hi = c["expect"]["range"]
                fig.add_trace(go.Scatter(x=[lo, hi], y=[c["select"]["point"]] * 2, mode="lines+markers",
                                         line={"color": "rgba(80,80,80,0.30)", "width": 12},
                                         marker={"symbol": "line-ns", "size": 16, "line": {"color": "#555", "width": 2}},
                                         name=f"limit of {h['id']} {c['id']}",
                                         hovertemplate=f"{h['id']} {c['id']}: {lo}..{hi} ppm<extra></extra>"),
                              row=1, col=k + 1)
nbfig.style(fig, "Clock drift of each point", "", "", height=max(380, 230 + 34 * len(cats)), margin={"t": 110})
fig.update_xaxes(zeroline=True, zerolinecolor="#6b6a64", zerolinewidth=1.5, title_text="drift (ppm)")
fig.update_yaxes(categoryorder="array", categoryarray=cats, autorange="reversed").show()

In [ ]:
dr.pivot_table(index="point", columns="experiment", values=["drift_ppm", "drift_per_sample"], sort=False).round(3)

## H2: the IRQ tail and the tick rate

H2 says that the SysTick ISRs make the IRQ tail. A periodic tick or a
higher tick rate gives more, shorter ISRs: a larger share of the tail.
A lower tick rate gives fewer, longer ISRs: a smaller share, but a
higher max. The CCDF gives the share of the edges with an IRQ entry of
x or more. The dashed lines are the points with 64 load timers. H2 a, f
and i are manual checks.

In [ ]:
if nbfig.built(inst, "irq-kconfig-1m"):
    nbfig.ccdf(nbfig.hists_of(inst, "irq-kconfig-1m", points=P0 + P64), "irq-kconfig-1m: CCDF of the IRQ entry", ["irq"],
               dashed=set(P64)).show()

In [ ]:
if nbfig.built(inst, "irq-kconfig-1m"):
    df = inst.rows("irq-kconfig-1m")
    out = df[(df.metric == "irq") & df.point.isin(P0 + P64)].set_index("point")[
        ["p50", "p90", "p99", "p99.9", "p99.99", "max", "mean"]]
else:
    out = None
out